# 01 · Inspección del límite provincial de Cusco

**Objetivo:** abrir `data/raw/Provincia/PROVINCIA.gpkg`, revisar su contenido y aislar la **provincia de Cusco** (no el departamento ni otras provincias).

Este notebook **solo lee** los datos; no modifica `data/raw/`.

**Clave del filtro:** el nombre `CUSCO` aparece dos veces: como departamento y como provincia. Por eso filtro por **códigos**:
- `ccdd = "08"` → departamento de Cusco
- `ccpp = "01"` → provincia de Cusco dentro de ese departamento

## 1. Localizar el archivo

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pyogrio


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la que contiene data/raw."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw").is_dir():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw desde esta ubicación.")


RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"

print("Raíz del proyecto:", RAIZ)
print("Archivo existe:", GPKG.exists())

## 2. Capas del GeoPackage

In [ ]:
# Cada fila: [nombre de la capa, tipo de geometría]
capas = pyogrio.list_layers(GPKG)
print("Capas encontradas:", len(capas))
for nombre, tipo in capas:
    print(f"  - {nombre}  ({tipo})")

## 3. Lectura, CRS, columnas y número de polígonos

In [ ]:
capa = capas[0][0]
gdf = gpd.read_file(GPKG, layer=capa)

print("Capa leída:", capa)
print("CRS:", gdf.crs)
print("Registros:", len(gdf))
print("Tipos de geometría:", gdf.geom_type.value_counts().to_dict())
print("Columnas:", list(gdf.columns))

gdf.drop(columns=gdf.geometry.name).head()

## 4. Revisar los códigos del departamento de Cusco

Así se ve qué provincias pertenecen al departamento (`ccdd = 08`) y cómo se llama cada una.

In [ ]:
print("Departamentos distintos:", gdf["ccdd"].nunique())
print("Provincias totales (Perú):", len(gdf))

gdf.loc[gdf["ccdd"] == "08", ["ccdd", "ccpp", "nombdep", "nombprov"]] \
    .sort_values("ccpp") \
    .reset_index(drop=True)

## 5. Filtrar la provincia de Cusco

Se espera **exactamente 1 registro**. Si no, el notebook se detiene.

In [ ]:
CCDD_CUSCO = "08"  # departamento de Cusco
CCPP_CUSCO = "01"  # provincia de Cusco dentro del departamento

prov = gdf[(gdf["ccdd"] == CCDD_CUSCO) & (gdf["ccpp"] == CCPP_CUSCO)].copy()
assert len(prov) == 1, f"Esperaba 1 provincia y encontré {len(prov)}"

prov[["ccdd", "ccpp", "nombdep", "nombprov", "fuente"]]

## 6. Validez de la geometría, partes y área

El área se calcula en **UTM zona 19S (EPSG:32719)**, la zona que corresponde a Cusco, porque el CRS del archivo es geográfico (grados).

In [ ]:
geom = prov.geometry.iloc[0]
partes = len(geom.geoms) if geom.geom_type == "MultiPolygon" else 1
area_km2 = prov.to_crs(epsg=32719).area.iloc[0] / 1e6

print("Tipo de geometría:", geom.geom_type)
print("Válida:", geom.is_valid, "| Vacía:", geom.is_empty)
print("Número de polígonos (partes):", partes)
print(f"Área aproximada: {area_km2:,.1f} km²  (EPSG:32719)")
print("Límites (lon/lat):", [round(v, 4) for v in prov.total_bounds])

## 7. Mapa de verificación

Izquierda: el departamento como contexto, con la provincia de Cusco resaltada.
Derecha: solo la provincia de Cusco.

In [ ]:
dpto = gdf[gdf["ccdd"] == CCDD_CUSCO]

fig, ejes = plt.subplots(1, 2, figsize=(14, 7))

dpto.plot(ax=ejes[0], color="#e9e9e9", edgecolor="#8a8a8a", linewidth=0.5)
prov.plot(ax=ejes[0], color="#1a6fc0", edgecolor="white", linewidth=0.8)
ejes[0].set_title("Departamento de Cusco (contexto)\nprovincia de Cusco en azul")

prov.plot(ax=ejes[1], color="#1a6fc0", edgecolor="white", linewidth=0.8)
ejes[1].set_title("Provincia de Cusco (ccdd=08, ccpp=01)")

for ax in ejes:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## Siguiente paso

Con el límite provincial ya verificado, el notebook `02_conexion_gee.ipynb` usa la misma geometría para conectarse a Google Earth Engine.